# Batch Evaluation — ROUGE-L & BERTScore

Quantitative evaluation of the fine-tuned adapter on the validation set,  
with per-sample score distributions and failure-case inspection.

**Kernel:** `gemma4_finetune` conda env  
**Prereqs:** `./data/vqa_val` and `./outputs/google_gemma-4-E2B-it_artifact_assessor_lora/`

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(repo_root, 'scripts'))
os.chdir(repo_root)

import torch
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_from_disk
from peft import PeftModel
from PIL import Image
from transformers import AutoProcessor, Gemma4ForConditionalGeneration

from utils import ADAPTER_PATH, DEVICE, DTYPE, MODEL_ID, USER_PROMPT

print(f'Device : {DEVICE} | dtype : {DTYPE}')

## Config

In [ ]:
MAX_SAMPLES  = 200   # set None to run the full val set (~4K samples)
MAX_NEW_TOKENS = 192
ADAPTER      = ADAPTER_PATH  # override with a different path if needed

## 1 — Load Model

In [ ]:
processor = AutoProcessor.from_pretrained(ADAPTER)

base = Gemma4ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map='auto' if DEVICE == 'cuda' else None,
    attn_implementation='sdpa',
)
model = PeftModel.from_pretrained(base, ADAPTER)
if DEVICE != 'cuda':
    model = model.to(DEVICE)
model.eval()
print('Model ready.')

## 2 — Load Validation Set

In [ ]:
val_ds = load_from_disk('./data/vqa_val')
if MAX_SAMPLES:
    val_ds = val_ds.select(range(min(MAX_SAMPLES, len(val_ds))))
print(f'Evaluating on {len(val_ds):,} samples')

## 3 — Run Inference

In [ ]:
from tqdm.notebook import tqdm

@torch.no_grad()
def assess_artifacts(image: Image.Image) -> str:
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': USER_PROMPT},
        ],
    }]
    text = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = processor(
        text=text, images=[[image]], return_tensors='pt'
    ).to(DEVICE)
    out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    generated = out_ids[0][inputs['input_ids'].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True)


refs, preds = [], []
for sample in tqdm(val_ds, desc='Inference'):
    refs.append(sample['assistant_response'])
    preds.append(assess_artifacts(sample['image']))

print(f'Generated {len(preds)} predictions.')

## 4 — ROUGE-L

In [ ]:
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = [
    rouge.score(ref, pred)['rougeL'].fmeasure
    for ref, pred in zip(refs, preds)
]

rouge_mean = np.mean(rouge_scores)
print(f'ROUGE-L mean : {rouge_mean:.4f}')
print(f'  Weak < 0.15 | Acceptable 0.15–0.30 | Strong > 0.30')

## 5 — BERTScore

In [ ]:
from bert_score import score as bert_score_fn

_, _, F1 = bert_score_fn(preds, refs, lang='en', device=DEVICE, verbose=False)
bert_scores = F1.numpy()

bert_mean = bert_scores.mean()
print(f'BERTScore F1 mean : {bert_mean:.4f}')
print(f'  Weak < 0.80 | Acceptable 0.80–0.88 | Strong > 0.88')

## 6 — Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(rouge_scores, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(rouge_mean, color='red', linestyle='--', label=f'mean={rouge_mean:.3f}')
axes[0].set_title('ROUGE-L Distribution')
axes[0].set_xlabel('ROUGE-L F1')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(bert_scores, bins=30, color='darkorange', edgecolor='white')
axes[1].axvline(bert_mean, color='red', linestyle='--', label=f'mean={bert_mean:.3f}')
axes[1].set_title('BERTScore F1 Distribution')
axes[1].set_xlabel('BERTScore F1')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7 — Summary Table

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    'Metric':    ['ROUGE-L', 'BERTScore F1'],
    'Mean':      [rouge_mean, bert_mean],
    'Std':       [np.std(rouge_scores), np.std(bert_scores)],
    'Min':       [np.min(rouge_scores), np.min(bert_scores)],
    'Median':    [np.median(rouge_scores), np.median(bert_scores)],
    'Max':       [np.max(rouge_scores), np.max(bert_scores)],
})
summary = summary.set_index('Metric').round(4)
display(summary)

## 8 — Failure Cases (Lowest ROUGE-L)

In [ ]:
N_WORST = 5

worst_idx = np.argsort(rouge_scores)[:N_WORST]

fig, axes = plt.subplots(1, N_WORST, figsize=(4 * N_WORST, 4))
if N_WORST == 1:
    axes = [axes]

for ax, idx in zip(axes, worst_idx):
    ax.imshow(val_ds[int(idx)]['image'])
    ax.axis('off')
    ax.set_title(f'#{idx} ROUGE={rouge_scores[idx]:.3f}')
plt.suptitle('Worst ROUGE-L samples', y=1.02)
plt.tight_layout()
plt.show()

for idx in worst_idx:
    print(f'--- Sample #{idx} | ROUGE-L={rouge_scores[idx]:.4f} | BERT={bert_scores[idx]:.4f} ---')
    print('GT  :', refs[idx])
    print('Pred:', preds[idx])
    print()

## 9 — Best Cases (Highest ROUGE-L)

In [ ]:
N_BEST = 5

best_idx = np.argsort(rouge_scores)[-N_BEST:][::-1]

fig, axes = plt.subplots(1, N_BEST, figsize=(4 * N_BEST, 4))
if N_BEST == 1:
    axes = [axes]

for ax, idx in zip(axes, best_idx):
    ax.imshow(val_ds[int(idx)]['image'])
    ax.axis('off')
    ax.set_title(f'#{idx} ROUGE={rouge_scores[idx]:.3f}')
plt.suptitle('Best ROUGE-L samples', y=1.02)
plt.tight_layout()
plt.show()

for idx in best_idx:
    print(f'--- Sample #{idx} | ROUGE-L={rouge_scores[idx]:.4f} | BERT={bert_scores[idx]:.4f} ---')
    print('GT  :', refs[idx])
    print('Pred:', preds[idx])
    print()

## 10 — ROUGE-L vs BERTScore Scatter

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(rouge_scores, bert_scores, alpha=0.4, s=15, color='steelblue')
plt.xlabel('ROUGE-L F1')
plt.ylabel('BERTScore F1')
plt.title('ROUGE-L vs BERTScore per sample')
plt.tight_layout()
plt.show()

corr = np.corrcoef(rouge_scores, bert_scores)[0, 1]
print(f'Pearson correlation: {corr:.4f}')

## 11 — Save Results to CSV

In [ ]:
results_df = pd.DataFrame({
    'reference':   refs,
    'prediction':  preds,
    'rouge_l':     rouge_scores,
    'bert_f1':     bert_scores,
})

out_path = './outputs/eval_results.csv'
results_df.to_csv(out_path, index=False)
print(f'Saved {len(results_df)} rows → {out_path}')